In [1]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    "openai:gpt-4o-mini",
    openai_api_key=os.getenv("openai_key"),
    temperature=0.7,
)

##  Memoria Sequenziale

In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
    ("system", "Act as a useful AI Assistant."),
    ("user", "{query}"),
])

# concatenazione del prompt al modello
chain = prompt | model

In [3]:
chain.invoke({"query": "Ciao, io mi chiamo Enzo"})

AIMessage(content='Ciao Enzo! Come posso aiutarti oggi?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 27, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb760GvzLF29IL6QSYvAMJ6rMKJzf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--96efc844-3e30-4b9b-abcc-ed6775f07c8a-0', usage_metadata={'input_tokens': 27, 'output_tokens': 11, 'total_tokens': 38, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
# notiamo che tutti gli LLM sono stateless, non hanno memoria
# e quindi non possono ricordare eventuali scambi di messaggi
# precedenti

chain.invoke({"query": "Come mi chiamo?"})

AIMessage(content='Non ho accesso alle informazioni personali degli utenti, quindi non posso sapere come ti chiami. Posso aiutarti con altre domande o informazioni di cui hai bisogno!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 23, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb76HOAb7Sq15utmpD4Hye8rmqfKn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a4bba6e3-45c9-45a5-978e-5268fcb48a1e-0', usage_metadata={'input_tokens': 23, 'output_tokens': 34, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

 Per permettere al modello di "ricordare" informazioni dalla conversazione, bisogna introdurre il concetto di __memoria__
 
Aggiungere la memoria significa, a conti fatti, iniettare lo storico della conversaione avuta sino a quel momento all'interno del prompt, così che ad ogni nuova richiesta che facciamo al LLM, questo riceva anche tutto lo scambio di conversazioni avute oltre che al SystemMessage e all'ultimo messaggio dell'utente

In [5]:
from langchain_core.prompts  import MessagesPlaceholder

prompt = ChatPromptTemplate([
    ("system", "Act as a useful AI Assistant. Answer using italian language"),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{query}"),
])

# -----------------------------------------------
# A regime il nostro prompt sarà strutturato così:
# > System message
# > History Message 1
# > History Message 2
# .
# .
# .
# > History Message N
# > Human Message (in input)
# -----------------------------------------------

chain = prompt | model

Facciamo una prova con una lista di messaggi fittizi che simulano uno storico di conversazione

In [6]:
dummy_messages = [
   ("user", "Ciao, io sono Simone"),
   ("assistant", "Ciao Simone, come posso aiutarti?"),
   ("user", "Volevo chiederti una cosa, posso?"),
   ("assistant", "Certamente! Dimmi pure"),
]

prompt.invoke({"history":dummy_messages, "query": "elencami i primi 3 pianeti del sistema solare"})

ChatPromptValue(messages=[SystemMessage(content='Act as a useful AI Assistant. Answer using italian language', additional_kwargs={}, response_metadata={}), HumanMessage(content='Ciao, io sono Simone', additional_kwargs={}, response_metadata={}), AIMessage(content='Ciao Simone, come posso aiutarti?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Volevo chiederti una cosa, posso?', additional_kwargs={}, response_metadata={}), AIMessage(content='Certamente! Dimmi pure', additional_kwargs={}, response_metadata={}), HumanMessage(content='elencami i primi 3 pianeti del sistema solare', additional_kwargs={}, response_metadata={})])

Ovviamente in un contesto reale, non mettiamo noi manualmente i messaggi in una lista, ma lasciamo che sia LangChain a farlo per noi.
In questo modo abbiamo anche la possibilità di tenere separate delle sessioni di conversazioni, ognuna con la sua memoria univoca

### RunnableWithMessageHistory

**Runnable che gestisce la cronologia dei messaggi di chat per un altro Runnable.**

Una *chat message history* è una sequenza di messaggi che rappresenta una conversazione.

`RunnableWithMessageHistory` incapsula un altro `Runnable` e gestisce per questo la *chat message history*; è responsabile della lettura e dell'aggiornamento della cronologia dei messaggi.

Deve sempre essere invocato con una configurazione (`config`) che contenga i parametri appropriati per il recupero dei messaggi corretti.

Per impostazione predefinita, il `Runnable` si aspetta un singolo parametro di configurazione chiamato `session_id` di tipo stringa; questo parametro viene utilizzato per creare una nuova *chat message history* o recuperarne una esistente associata al `session_id` specificato.


In [7]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

chat_map = {}

def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]


chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history"
)

In [8]:
chain_with_history.invoke(
    {"query": "Ciao, mi chiamo Enzo"},
    config={"session_id": "id_1234"}
)

AIMessage(content='Ciao Enzo! Come posso aiutarti oggi?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 30, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb7CtoCs0Ojw4RNnsTMPQD3LTsakA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--3b4de748-06a8-4aa6-be84-23af4bb6db5d-0', usage_metadata={'input_tokens': 30, 'output_tokens': 11, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
chat_map["id_1234"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='Ciao, mi chiamo Enzo', additional_kwargs={}, response_metadata={}), AIMessage(content='Ciao Enzo! Come posso aiutarti oggi?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 30, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb7CtoCs0Ojw4RNnsTMPQD3LTsakA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--3b4de748-06a8-4aa6-be84-23af4bb6db5d-0', usage_metadata={'input_tokens': 30, 'output_tokens': 11, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [10]:
chain_with_history.invoke(
    {"query": "Ciao, mi chiamo Simone"},
    config={"session_id": "id_5678"}
)

AIMessage(content='Ciao Simone! Come posso aiutarti oggi?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 29, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb7DY14OIqyxR1sHStmD2ARFoIr0s', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--469deeaf-db28-4b33-9d2e-be2fdc09340e-0', usage_metadata={'input_tokens': 29, 'output_tokens': 10, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [11]:
chat_map["id_5678"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='Ciao, mi chiamo Simone', additional_kwargs={}, response_metadata={}), AIMessage(content='Ciao Simone! Come posso aiutarti oggi?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 29, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb7DY14OIqyxR1sHStmD2ARFoIr0s', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--469deeaf-db28-4b33-9d2e-be2fdc09340e-0', usage_metadata={'input_tokens': 29, 'output_tokens': 10, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [12]:
chain_with_history.invoke(
    {"query": "Come mi chiamo?" },
    config={"session_id": "id_1234"}
)

AIMessage(content='Ti chiami Enzo! Se hai bisogno di ulteriore aiuto o informazioni, fammelo sapere!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 54, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb7DusgaRQNsG1fN0GgTXhymkKAb2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a017bf92-4653-4415-8b2d-d5ec3b6f41ed-0', usage_metadata={'input_tokens': 54, 'output_tokens': 23, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
chain_with_history.invoke(
    {"query": "Come mi chiamo?" },
    config={"session_id": "id_5678"}
)

AIMessage(content='Ti chiami Simone! Come posso assisterti?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 52, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb7EEOKXQ8D6SiUPhHDbJMakXTJ1u', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--b6ddfc17-3c5b-49ec-97bc-d2953083ed1f-0', usage_metadata={'input_tokens': 52, 'output_tokens': 10, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})